# 03. Цепи и классы толщины

| вход | выход |
|---|---|
| маска штрихов и маска текста из этапа 02 | маска сохраненных линий `clean_mask` |

Скелет рвет линию на ребра в каждом пересечении, поэтому толщина короткого ребра шумит. Цепь
собирает ребра обратно: концы в одном узле склеиваются, если продолжают друг друга по направлению,
лежат на одной прямой и близки по толщине.

Границы классов толщины не задаются руками, а находятся кластеризацией по самому кадру. Текстовые
цепи в обучении не участвуют: буквы тонкие и многочисленные, они стягивают границу вниз и уводят
часть контура в младший класс.

Итоговая маска не вычитается из исходной, а рисуется заново от осей сохраненных цепей.

In [1]:
from pathlib import Path

from artifact_cache import ArtifactCache
from ipywidgets import Checkbox, Dropdown, FloatSlider, IntSlider, interactive
from stage_metrics import (
    ChainMetrics,
    ThicknessClassMetrics,
    measure_chain_metrics,
    measure_mask_metrics,
    measure_thickness_class_metrics,
)
from visualization import build_view_window, draw_chains, draw_thickness_classes, show_mask, window_sliders

from draftslice.chain_merging import ChainMergeParameters, MergeGates, StrokeChains, build_stroke_chains
from draftslice.common_types import Mask, Mat
from draftslice.image_preparation import load_bgr_image, scale_image
from draftslice.stroke_graph import StrokeGraph, build_stroke_graph
from draftslice.stroke_painting import paint_selected_paths
from draftslice.thickness_classes import ThicknessClasses, fit_thickness_classes, select_chains_by_class

DATASET_FOLDER = Path("/mnt/data/datasets/Сравнение чертежей/Эскизы датасет")
TEXT_SHARE_LIMIT = 0.5

## Вход из этапа 02

In [2]:
cache = ArtifactCache()
source_files: list[Path] = [path for path in sorted(DATASET_FOLDER.glob("*.jpg")) if cache.has_stage(path, "strokes")]
print(f"файлов с посчитанной маской штрихов: {len(source_files)}")


def run_source(file_index: int) -> Path:
    """Выбор файла среди посчитанных на этапе 02."""
    source_path = source_files[file_index]
    stored = cache.load_latest(source_path, "strokes")
    print(f"артефакт: {stored.path.name}")
    print(f"параметры этапа 02: {stored.parameters}")
    return source_path


ui_source = interactive(
    run_source,
    file_index=Dropdown(
        options=[(f"[{index}] {path.stem}", index) for index, path in enumerate(source_files)],
        value=0,
        description="файл",
    ),
)
ui_source

файлов с посчитанной маской штрихов: 3


interactive(children=(Dropdown(description='файл', options=(('[0] 1', 0), ('[1] 13', 1), ('[2] 9', 2)), value=…

In [3]:
source_path: Path = ui_source.result
stored_strokes = cache.load_latest(source_path, "strokes")
working_scale: float = float(stored_strokes.parameters["working_scale"])

strokes_mask: Mask = stored_strokes.arrays["strokes_mask"]
text_region_mask: Mask = stored_strokes.arrays["text_region_mask"]
scaled_image: Mat = scale_image(load_bgr_image(source_path), working_scale)

graph: StrokeGraph = build_stroke_graph(strokes_mask, text_region_mask)
print(f"кадр {scaled_image.shape[1]}x{scaled_image.shape[0]} | масштаб x{working_scale:.0f} | ребер {graph.path_count}")
print(f"медиана толщины {graph.median_thickness:.1f} px")

кадр 2950x1838 | масштаб x2 | ребер 1833
медиана толщины 3.0 px


## Склейка ребер в цепи

In [4]:
def report_chain_metrics(metrics: ChainMetrics) -> None:
    """Печать метрик склейки."""
    print(
        f"цепей {metrics.chain_count} | звеньев {metrics.linked_path_count} | "
        f"звеньев на цепь {metrics.links_per_chain:.2f}"
    )
    print(
        f"длина: медиана {metrics.median_length:.0f}, максимум {metrics.maximum_length:.0f} px | "
        f"текстовых цепей {metrics.text_chain_count} | без толщины {metrics.undefined_thickness_count}"
    )


def run_chains(
    center_x: float,
    center_y: float,
    zoom: float,
    node_radius: float,
    tangent_length: int,
    maximum_deviation: float,
    maximum_offset: float,
    thickness_ratio: float,
) -> StrokeChains:
    """Слайдеры склейки ребер в цепи."""
    parameters = ChainMergeParameters(
        node_radius_pixels=node_radius,
        tangent_length_pixels=tangent_length,
        gates=MergeGates(maximum_deviation, maximum_offset, thickness_ratio),
    )
    chains = build_stroke_chains(graph, parameters)
    report_chain_metrics(measure_chain_metrics(chains, TEXT_SHARE_LIMIT))
    draw_chains(
        graph,
        chains,
        scaled_image,
        build_view_window(scaled_image.shape[:2], center_x, center_y, zoom),
        f"цепи | узел {node_radius:.1f} px, угол < {maximum_deviation:.0f}, сдвиг < {maximum_offset:.1f} px",
    )
    return chains


ui_chains = interactive(
    run_chains,
    **window_sliders(),
    node_radius=FloatSlider(value=12.0, min=0.0, max=20.0, step=0.5, continuous_update=False, description="узел, px"),
    tangent_length=IntSlider(value=20, min=3, max=40, step=1, continuous_update=False, description="касат., px"),
    maximum_deviation=FloatSlider(value=20.0, min=0.0, max=45.0, step=1.0, continuous_update=False, description="угол"),
    maximum_offset=FloatSlider(value=2.5, min=0.0, max=20.0, step=0.5, continuous_update=False, description="сдвиг"),
    thickness_ratio=FloatSlider(
        value=1.2, min=1.0, max=3.0, step=0.1, continuous_update=False, description="t max/min"
    ),
)
ui_chains

interactive(children=(FloatSlider(value=0.5, continuous_update=False, description='центр X', max=1.0, step=0.0…

In [5]:
chains: StrokeChains = ui_chains.result
print(f"цепей {chains.chain_count} | звеньев {len(chains.path_indices)}")

цепей 1262 | звеньев 1636


## Классы толщины

In [6]:
def report_class_metrics(metrics: ThicknessClassMetrics, working_scale: float) -> None:
    """Печать метрик классов толщины."""
    print(f"центры, px:  {metrics.centers.round(2)} | в оригинале {(metrics.centers / working_scale).round(2)}")
    print(f"границы, px: {metrics.edges.round(2)} | в оригинале {(metrics.edges / working_scale).round(2)}")
    for class_index, (count, share) in enumerate(zip(metrics.chain_counts, metrics.length_shares, strict=True)):
        print(f"класс {class_index}: цепей {count} | длины {share:.1%}")
    print(f"без класса: {metrics.unclassified_count} цепей")


def run_classes(
    center_x: float, center_y: float, zoom: float, class_count: int, weight_by_length: bool
) -> ThicknessClasses:
    """Слайдер числа классов толщины."""
    classes = fit_thickness_classes(chains, class_count, TEXT_SHARE_LIMIT, weight_by_length)
    report_class_metrics(measure_thickness_class_metrics(chains, classes), working_scale)
    draw_thickness_classes(
        graph,
        chains,
        classes,
        scaled_image,
        build_view_window(scaled_image.shape[:2], center_x, center_y, zoom),
        f"{class_count} классов | границы {classes.edges.round(1)} px | серое это текст",
    )
    return classes


ui_classes = interactive(
    run_classes,
    **window_sliders(),
    class_count=IntSlider(value=2, min=2, max=6, step=1, continuous_update=False, description="классов"),
    weight_by_length=Checkbox(value=True, description="вес по длине"),
)
ui_classes

interactive(children=(FloatSlider(value=0.5, continuous_update=False, description='центр X', max=1.0, step=0.0…

In [7]:
classes: ThicknessClasses = ui_classes.result
print(f"классов {classes.class_count} | цепей без класса {int((classes.label < 0).sum())}")

классов 2 | цепей без класса 658


## Перерисовка сохраненных линий

Удалять пиксели нельзя: там, где тонкая линия упирается в контур, часть пикселей контура ближе к оси
тонкой линии, и наивное вычитание выкусывает из контура клин. Поэтому сохраненные линии рисуются
заново от своих осей, а удаленное просто не рисуется.

In [8]:
def run_clean_mask(minimum_class: int, radius_tolerance: float) -> Mask:
    """Слайдеры итоговой маски этапа."""
    kept_chains = select_chains_by_class(chains, classes, minimum_class, TEXT_SHARE_LIMIT)
    kept_paths = chains.expand_selection_to_paths(kept_chains)
    mask = paint_selected_paths(graph, kept_paths, strokes_mask, radius_tolerance)

    metrics = measure_mask_metrics(mask)
    print(f"цепей {int(kept_chains.sum())}/{chains.chain_count} | ребер {int(kept_paths.sum())}/{graph.path_count}")
    print(
        f"чернил {mask.sum() / strokes_mask.sum():.1%} от бинаризации | "
        f"кусков {metrics.component_count} | пустот {metrics.hole_count}"
    )
    show_mask(mask, f"классы >= {minimum_class}, допуск {radius_tolerance:.1f} px")
    return mask


ui_clean_mask = interactive(
    run_clean_mask,
    minimum_class=IntSlider(value=1, min=0, max=5, step=1, continuous_update=False, description="класс >="),
    radius_tolerance=FloatSlider(value=0.5, min=0.0, max=2.0, step=0.1, continuous_update=False, description="допуск"),
)
ui_clean_mask

interactive(children=(IntSlider(value=1, continuous_update=False, description='класс >=', max=5), FloatSlider(…

## Артефакт для следующего этапа

In [9]:
clean_mask: Mask = ui_clean_mask.result
stage_parameters = {
    "source_stage": stored_strokes.path.name,
    "text_share_limit": TEXT_SHARE_LIMIT,
    **{name: value for name, value in ui_chains.kwargs.items() if name not in ("center_x", "center_y", "zoom")},
    **{name: value for name, value in ui_classes.kwargs.items() if name not in ("center_x", "center_y", "zoom")},
    **ui_clean_mask.kwargs,
}

saved_path = cache.save(
    source_path,
    "clean_mask",
    stage_parameters,
    clean_mask=clean_mask,
    text_region_mask=text_region_mask,
    thickness_class_edges=classes.edges,
)
print(f"сохранено: {saved_path}")
print(f"параметры: {stage_parameters}")

сохранено: .cache/9/clean_mask__e4c8359283.npz
параметры: {'source_stage': 'strokes__6c39af70c3.npz', 'text_share_limit': 0.5, 'node_radius': 12.0, 'tangent_length': 20, 'maximum_deviation': 20.0, 'maximum_offset': 2.5, 'thickness_ratio': 1.2, 'class_count': 2, 'weight_by_length': True, 'minimum_class': 1, 'radius_tolerance': 0.5}
